In [ ]:
# ====================================================
# Full modeling pipeline: LightGBM + MLP + Stacking
# NOW WITH TEST SET PREDICTION
# ====================================================
# Requirements: numpy, pandas, sklearn, lightgbm, torch, tqdm
# Run in Colab or local Python environment.
# ====================================================

import os, gc, math, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import re

# ---------------------------
# Config - edit to taste
# ---------------------------
### PATHS FOR TRAINING DATA
TRAIN_EMB_PATH = "/kaggle/input/embedding-mpz/aggregated_embeddings.npz" # aggregated embeddings file
TRAIN_CSV = "/kaggle/input/train-csv/train.csv" # original CSV for text features if needed

### ADDED: PATHS FOR TEST DATA
TEST_EMB_PATH = "/kaggle/input/test-fast-embeding/all_embeddings_aggregated.npz" # <-- IMPORTANT: UPDATE THIS PATH
TEST_CSV = "/kaggle/input/test1-csv/test.csv" # <-- IMPORTANT: UPDATE THIS PATH

### OUTPUT DIRECTORY AND SUBMISSION FILENAME
OUT_DIR = "/kaggle/working/models_and_preds"
SUBMISSION_CSV = os.path.join(OUT_DIR, "submission.csv")
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
N_FOLDS = 5
RANDOM_STATE = SEED

# PCA dims for modalities (tunable)
PCA_TEXT_DIM = 256
PCA_IMG_DIM = 256

# LightGBM CV params (balanced speed/quality)
LGB_PARAMS = dict(
    objective="regression",
    metric="rmse",
    learning_rate=0.03,
    num_leaves=64,
    max_depth=-1,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    min_data_in_leaf=20,
    lambda_l1=0.0,
    lambda_l2=1.0,
    n_jobs=-1,
    verbosity=-1,
    seed=SEED
)

# MLP training params
MLP_EPOCHS = 50
MLP_BATCH = 256
MLP_LR = 3e-4
MLP_HIDDEN = [512, 256]

# Utility: SMAPE
def smape(y_true, y_pred, eps=1e-8):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return float(np.mean(200.0 * np.abs(y_pred - y_true) / (np.abs(y_pred) + np.abs(y_true) + eps)))

# ---------------------------
# Feature Engineering Functions (to be used on both train and test)
# ---------------------------
def extract_size_and_unit(text):
    text = str(text).lower()
    m = re.search(r'(\d+(?:\.\d+)?)\s*(oz|ounce|ml|l|litre|liter|g|gram|kg|pack|ct|count|pc|pcs)\b', text)
    if m:
        val, unit = float(m.group(1)), m.group(2)
        if unit in ['pc','pcs']: unit='count'
        if unit in ['litre','liter']: unit='l'
        if unit in ['ounce','ounces']: unit='oz'
        if unit in ['gram']: unit='g'
        return val, unit
    return 0.0, "none"

def extract_pack_count(text):
    m = re.search(r'(\d+)\s*(pack|count|ct|pc|pcs)\b', str(text).lower())
    return int(m.group(1)) if m else 1

def create_numeric_features(df_meta, top_units=None):
    if df_meta is not None and "catalog_content" in df_meta.columns:
        texts = df_meta["catalog_content"].astype(str).tolist()
    else:
        texts = [""] * len(df_meta)

    size_vals, size_units, pack_counts = [], [], []
    for t in texts:
        v,u = extract_size_and_unit(t)
        size_vals.append(v)
        size_units.append(u)
        pack_counts.append(extract_pack_count(t))

    total_size = []
    for v,u,p in zip(size_vals, size_units, pack_counts):
        if p>1 and v>0:
            total_size.append(p*v if u not in ['pack','count'] else p)
        else:
            total_size.append(max(v, p, 1.0))

    num_df = pd.DataFrame({
        "size_val": np.array(size_vals, dtype=np.float32),
        "pack_count": np.array(pack_counts, dtype=np.float32),
        "total_size": np.array(total_size, dtype=np.float32),
    })

    if top_units is None:
        top_units = pd.Series(size_units).value_counts().index[:10].tolist()
    
    for u in top_units:
        num_df[f"unit_{u}"] = (np.array(size_units) == u).astype(np.float32)
    return num_df, top_units

# ---------------------------
# Load and Process TRAINING Data
# ---------------------------
print("Loading TRAIN embeddings from:", TRAIN_EMB_PATH)
data = np.load(TRAIN_EMB_PATH, allow_pickle=True)
text_emb = data["text"]
img_emb  = data["image"]
y_all    = data["price"]
ids_all  = data["sample_id"] if "sample_id" in data else np.arange(len(y_all))

N = len(y_all)
print(f"Loaded N={N} train samples | text:{text_emb.shape} image:{img_emb.shape}")

log_price = np.log1p(y_all)
low_q, high_q = np.quantile(log_price, 0.025), np.quantile(log_price, 0.975)
keep_mask = (log_price >= low_q) & (log_price <= high_q)
print(f"Trimming log-price tails outside [{low_q:.4f}, {high_q:.4f}] -> keep {keep_mask.sum()}/{N}")

text_emb = text_emb[keep_mask]
img_emb  = img_emb[keep_mask]
y_all    = y_all[keep_mask]
ids_all  = ids_all[keep_mask]
N = len(y_all)

df_meta_train = pd.read_csv(TRAIN_CSV).iloc[:len(data["price"])][keep_mask]
num_df_train, top_units = create_numeric_features(df_meta_train)

text_norm = np.linalg.norm(text_emb, axis=1).reshape(-1,1)
img_norm  = np.linalg.norm(img_emb, axis=1).reshape(-1,1)
extra_df_train = pd.DataFrame(np.hstack([text_norm, img_norm]), columns=["t_norm","i_norm"])

sc_text = StandardScaler()
sc_img  = StandardScaler()
text_s = sc_text.fit_transform(text_emb)
img_s  = sc_img.fit_transform(img_emb)

pca_t = PCA(n_components=min(PCA_TEXT_DIM, text_s.shape[1]), random_state=SEED)
pca_i = PCA(n_components=min(PCA_IMG_DIM, img_s.shape[1]), random_state=SEED)
text_p = pca_t.fit_transform(text_s)
img_p  = pca_i.fit_transform(img_s)

X_numeric = np.hstack([num_df_train.values, extra_df_train.values]).astype(np.float32)
X = np.hstack([text_p, img_p, X_numeric]).astype(np.float32)
y = y_all.copy()
print("Final TRAIN X shape:", X.shape)


# ---------------------------
### ADDED: Load and Process TEST Data
# ---------------------------
print("\nLoading TEST embeddings from:", TEST_EMB_PATH)
test_data = np.load(TEST_EMB_PATH, allow_pickle=True)
text_emb_test = test_data["text"]
img_emb_test  = test_data["image"]
ids_test      = test_data["sample_id"] if "sample_id" in test_data else np.arange(len(text_emb_test))
print(f"Loaded N={len(ids_test)} test samples | text:{text_emb_test.shape} image:{img_emb_test.shape}")

df_meta_test = pd.read_csv(TEST_CSV)
num_df_test, _ = create_numeric_features(df_meta_test, top_units=top_units)

text_norm_test = np.linalg.norm(text_emb_test, axis=1).reshape(-1,1)
img_norm_test  = np.linalg.norm(img_emb_test, axis=1).reshape(-1,1)
extra_df_test = pd.DataFrame(np.hstack([text_norm_test, img_norm_test]), columns=["t_norm","i_norm"])

# IMPORTANT: Use .transform() with scalers and PCA fitted on training data
text_s_test = sc_text.transform(text_emb_test)
img_s_test  = sc_img.transform(img_emb_test)
text_p_test = pca_t.transform(text_s_test)
img_p_test  = pca_i.transform(img_s_test)

X_numeric_test = np.hstack([num_df_test.values, extra_df_test.values]).astype(np.float32)
X_test = np.hstack([text_p_test, img_p_test, X_numeric_test]).astype(np.float32)
print("Final TEST X shape:", X_test.shape)


# ---------------------------
# KFold OOF: LightGBM and MLP
# ---------------------------
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
oof_lgb = np.zeros(len(y))
oof_mlp = np.zeros(len(y))

print("\n=== LightGBM OOF training ===")
# ... (LGBM training remains unchanged)
for fold, (tr_idx, va_idx) in enumerate(kf.split(X, y)):
    print(f"\n-- LGB Fold {fold+1}/{N_FOLDS}")
    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = np.log1p(y[tr_idx]), np.log1p(y[va_idx])
    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dvalid = lgb.Dataset(X_va, label=y_va)
    bst = lgb.train(
        LGB_PARAMS, dtrain, num_boost_round=5000,
        valid_sets=[dtrain, dvalid], valid_names=["train","valid"],
        callbacks=[lgb.early_stopping(200), lgb.log_evaluation(500)]
    )
    pred_va = np.expm1(bst.predict(X_va, num_iteration=bst.best_iteration))
    oof_lgb[va_idx] = pred_va
    gc.collect()
print("LGB OOF SMAPE:", smape(y, oof_lgb))




Loading TRAIN embeddings from: /kaggle/input/embedding-mpz/aggregated_embeddings.npz
Loaded N=70000 train samples | text:(70000, 384) image:(70000, 768)
Trimming log-price tails outside [1.2208, 4.3497] -> keep 66506/70000
Final TRAIN X shape: (66506, 526)

Loading TEST embeddings from: /kaggle/input/test-fast-embeding/all_embeddings_aggregated.npz
Loaded N=75000 test samples | text:(75000, 384) image:(75000, 768)
Final TEST X shape: (75000, 526)

=== LightGBM OOF training ===

-- LGB Fold 1/5
Training until validation scores don't improve for 200 rounds
[500]	train's rmse: 0.463755	valid's rmse: 0.622215
[1000]	train's rmse: 0.353069	valid's rmse: 0.613886
[1500]	train's rmse: 0.275508	valid's rmse: 0.610243
[2000]	train's rmse: 0.218524	valid's rmse: 0.608697
[2500]	train's rmse: 0.174133	valid's rmse: 0.607518
[3000]	train's rmse: 0.14004	valid's rmse: 0.606943
[3500]	train's rmse: 0.113037	valid's rmse: 0.606231
[4000]	train's rmse: 0.0921398	valid's rmse: 0.605888
[4500]	train's r

In [2]:
# ---------------------------
# Retrain on full data and Predict
# ---------------------------
import lightgbm as lgb
import numpy as np
import pandas as pd

print("\nRetraining final LGB model on full dataset...")

# Create the full training dataset for LightGBM
dtrain_full = lgb.Dataset(X, label=np.log1p(y))

# Determine the optimal number of rounds from the K-Fold training.
# 'bst' is the model from the last fold, which stores the best iteration.
best_rounds = bst.best_iteration if 'bst' in locals() and bst.best_iteration > 0 else 1000

# Train the final model on all data
bst_full = lgb.train(LGB_PARAMS, dtrain_full, num_boost_round=best_rounds)

print("Generating predictions on the test set...")
# Predict on the test data and convert back from log scale
final_preds = np.expm1(bst_full.predict(X_test))


# ---------------------------
# Create Submission File
# ---------------------------
print("\nCreating submission file...")
submission_df = pd.DataFrame({'sample_id': ids_test, 'price': final_preds})

# It's good practice to ensure predictions are not negative
submission_df['price'] = submission_df['price'].clip(lower=0)

# Save the final CSV
submission_df.to_csv(SUBMISSION_CSV, index=False)

print(f"\nSubmission file created at: {SUBMISSION_CSV}")
print("Top 5 predictions:")
print(submission_df.head())

# Print the final OOF score you calculated earlier
oof_score = smape(y, oof_lgb)
print(f"\nDone. Final LGB OOF SMAPE: {oof_score:.3f}")

Generating predictions on the test set...

Creating submission file...

Submission file created at: /kaggle/working/models_and_preds/submission.csv
Top 5 predictions:
   sample_id      price
0          0  10.959971
1          1  20.908907
2          2  35.663202
3          3  15.580572
4          4   7.350955

Done. Final LGB OOF SMAPE: 48.710


In [4]:
# ---------------------------
# Retrain on full data and Predict
# ---------------------------
import lightgbm as lgb
import numpy as np
import pandas as pd

print("\nRetraining final LGB model on full dataset...")

# Create the full training dataset for LightGBM
dtrain_full = lgb.Dataset(X, label=np.log1p(y))

# Determine the optimal number of rounds from the K-Fold training.
# 'bst' is the model from the last fold, which stores the best iteration.
best_rounds = bst.best_iteration if 'bst' in locals() and bst.best_iteration > 0 else 1000

# Train the final model on all data
bst_full = lgb.train(LGB_PARAMS, dtrain_full, num_boost_round=best_rounds)


# ---------------------------
# Save the Trained Model
# ---------------------------
print("\nSaving the trained model...")
# Define the filename for the saved model
MODEL_PATH = "/kaggle/working/model-amazon"

# Save the model to a file
bst_full.save_model(MODEL_PATH)

print(f"Model successfully saved to: {MODEL_PATH}")


# ---------------------------
# Generate Predictions
# ---------------------------
print("\nGenerating predictions on the test set...")
# Predict on the test data and convert back from log scale
final_preds = np.expm1(bst_full.predict(X_test))


# ---------------------------
# Create Submission File
# ---------------------------
print("\nCreating submission file...")
submission_df = pd.DataFrame({'sample_id': ids_test, 'price': final_preds})

# It's good practice to ensure predictions are not negative
submission_df['price'] = submission_df['price'].clip(lower=0)

# Save the final CSV
submission_df.to_csv(SUBMISSION_CSV, index=False)

print(f"\nSubmission file created at: {SUBMISSION_CSV}")
print("Top 5 predictions:")
print(submission_df.head())

# Print the final OOF score you calculated earlier
oof_score = smape(y, oof_lgb)
print(f"\nDone. Final LGB OOF SMAPE: {oof_score:.3f}")


Retraining final LGB model on full dataset...

Saving the trained model...
Model successfully saved to: /kaggle/working/model-amazon

Generating predictions on the test set...

Creating submission file...

Submission file created at: /kaggle/working/models_and_preds/submission.csv
Top 5 predictions:
   sample_id      price
0          0  10.959971
1          1  20.908907
2          2  35.663202
3          3  15.580572
4          4   7.350955

Done. Final LGB OOF SMAPE: 48.710
